# 03 — Comparative Analysis (Classical vs Quantum GAN)

**ZHAW CEC Quantum Computing — Final project**

Loads `metrics.json` files from every run produced by notebooks 01 and 02, assembles them into a tidy table, and produces the comparison figures used in the final presentation.

**No model training in this notebook.** Pure analysis of saved results. Re-run anytime new results land in `results/`.

## Setup

In [ ]:
import os
if not os.path.isdir('/content/qGAN-market-generator'):
    !git clone https://github.com/wuns/qGAN-market-generator.git
%cd /content/qGAN-market-generator

In [ ]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd()
if (ROOT / 'src').is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / 'src').is_dir():
    sys.path.insert(0, str(ROOT.parent))
    ROOT = ROOT.parent

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = ROOT / 'results'
print('Looking for metrics in:', RESULTS)

## 1. Discover all completed runs

Scan `results/` for every subdirectory containing a `metrics.json`. Each one becomes a row.

In [ ]:
def parse_run_folder(folder_name: str) -> dict:
    """Decode the folder naming convention into structured fields.

    Folders look like:
      classical_gan_SSMI
      classical_wgan_gp_SSMI_GDAXI
      quantum_wgan_gp_SSMI_GDAXI_q6L3
    """
    parts = folder_name.split('_')
    family = parts[0]                                  # 'classical' or 'quantum'
    # variant: 'gan' or 'wgan_gp' — figure out by checking against known options
    if 'wgan' in folder_name:
        variant = 'wgan_gp'
    else:
        variant = 'gan'
    # Strip family + variant prefix to find the asset block (and quantum suffix)
    suffix = folder_name
    for prefix in [f'{family}_{variant}_', f'{family}_']:
        if suffix.startswith(prefix):
            suffix = suffix[len(prefix):]
            break
    qubits = layers = None
    asset_part = suffix
    if '_q' in suffix and 'L' in suffix.split('_q')[-1]:
        asset_part, q_suffix = suffix.rsplit('_q', 1)
        try:
            qubits, layers = q_suffix.split('L')
            qubits, layers = int(qubits), int(layers)
        except ValueError:
            pass
    return {
        'family':  family,
        'variant': variant,
        'assets':  asset_part,
        'n_qubits': qubits,
        'n_layers': layers,
    }


rows = []
for folder in sorted(RESULTS.iterdir() if RESULTS.is_dir() else []):
    metrics_file = folder / 'metrics.json'
    if not metrics_file.is_file():
        continue
    with open(metrics_file) as f:
        m = json.load(f)
    parsed = parse_run_folder(folder.name)
    row = {'run': folder.name, **parsed}

    # Flatten the metrics fields we care about
    row['ks']                      = m.get('ks_statistic_overall')
    row['training_time_sec']       = m.get('training_time_sec')
    row['n_params_G']              = m.get('n_params_generator')
    row['n_params_D']              = m.get('n_params_discriminator')
    if 'dependence' in m:
        row['bures_wasserstein']   = m['dependence'].get('bures_wasserstein')
        row['frobenius_cov']       = m['dependence'].get('frobenius_cov_err')
        row['frobenius_corr']      = m['dependence'].get('frobenius_corr_err')
    elif 'correlation' in m:        # backwards-compat for older runs
        row['frobenius_corr']      = m['correlation'].get('frobenius_err')
    # Per-asset stats: collapse to mean across assets for the table
    if 'real_per_asset' in m:
        kurt_real = np.mean([v['kurtosis'] for v in m['real_per_asset'].values()])
        kurt_fake = np.mean([v['kurtosis'] for v in m['fake_per_asset'].values()])
        std_real  = np.mean([v['std']      for v in m['real_per_asset'].values()])
        std_fake  = np.mean([v['std']      for v in m['fake_per_asset'].values()])
        row['kurtosis_real'] = kurt_real
        row['kurtosis_fake'] = kurt_fake
        row['kurtosis_gap']  = abs(kurt_real - kurt_fake)
        row['std_real']      = std_real
        row['std_fake']      = std_fake
        row['std_rel_err']   = abs(std_real - std_fake) / std_real
    rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
    print('No metrics files found in results/. Run notebooks 01 and 02 first.')
else:
    print(f'Found {len(df)} runs:')
    print(df[['family', 'variant', 'assets', 'n_qubits', 'n_layers']].to_string(index=False))

## 2. Summary table

The numbers behind every figure on your slides. Sorted with classical baselines first, then quantum runs.

In [ ]:
if df.empty:
    print('No data — skipping.')
else:
    cols = ['family', 'variant', 'assets', 'n_qubits', 'n_layers',
            'n_params_G', 'training_time_sec',
            'ks', 'std_rel_err', 'kurtosis_gap']
    if 'bures_wasserstein' in df.columns:
        cols.append('bures_wasserstein')
    if 'frobenius_corr' in df.columns:
        cols.append('frobenius_corr')
    summary = df[[c for c in cols if c in df.columns]].copy()
    summary = summary.sort_values(['assets', 'family', 'variant', 'n_qubits'])
    summary.to_csv(RESULTS / 'summary.csv', index=False)
    print('Saved summary table to results/summary.csv')
    summary

## 3. Headline comparison: classical vs quantum on shared metrics

For each combination of `(variant, assets)` that has both a classical and a quantum run, plot the comparison side by side. Lower is better for all metrics shown.

The metrics on the bar chart are normalized within each panel so the bars are comparable visually — the actual values are in the summary table above.

In [ ]:
if df.empty:
    print('No data — skipping.')
else:
    # Identify pairs: same (variant, assets) where both classical and quantum exist.
    paired_keys = []
    for (variant, assets), sub in df.groupby(['variant', 'assets']):
        families = set(sub['family'])
        if {'classical', 'quantum'}.issubset(families):
            paired_keys.append((variant, assets))
    print(f'Found {len(paired_keys)} (variant, assets) pairs with both families:')
    for k in paired_keys:
        print(' ', k)

In [ ]:
if not df.empty and paired_keys:
    metric_cols = [c for c in ['ks', 'std_rel_err', 'kurtosis_gap',
                                'bures_wasserstein', 'frobenius_corr']
                   if c in df.columns]
    n_metrics = len(metric_cols)

    fig, axes = plt.subplots(len(paired_keys), 1,
                              figsize=(2 + 1.6 * n_metrics, 3 * len(paired_keys)),
                              squeeze=False)

    for ax_row, (variant, assets) in zip(axes[:, 0], paired_keys):
        sub = df[(df['variant'] == variant) & (df['assets'] == assets)]
        # Use the highest-qubit run when multiple quantum entries exist
        classical = sub[sub['family'] == 'classical'].iloc[0]
        quantum   = sub[sub['family'] == 'quantum'].sort_values('n_qubits',
                                                                  na_position='first').iloc[-1]

        # Normalize each metric to [0, 1] across this pair so bars are visible
        x = np.arange(n_metrics)
        width = 0.35
        cls_vals = np.array([classical.get(c) for c in metric_cols], dtype=float)
        qnt_vals = np.array([quantum.get(c)   for c in metric_cols], dtype=float)
        denom = np.maximum(cls_vals, qnt_vals)
        denom = np.where(denom == 0, 1.0, denom)
        cls_norm = cls_vals / denom
        qnt_norm = qnt_vals / denom

        ax_row.bar(x - width/2, cls_norm, width, label=f'classical ({variant})')
        ax_row.bar(x + width/2, qnt_norm, width,
                   label=f"quantum ({variant}, q={int(quantum['n_qubits']) if pd.notna(quantum['n_qubits']) else '?'})")
        ax_row.set_xticks(x); ax_row.set_xticklabels(metric_cols, rotation=20)
        ax_row.set_ylabel('normalized\n(0=better)')
        ax_row.set_title(f'assets: {assets}  |  variant: {variant}')
        ax_row.legend(loc='upper right', fontsize=9)
        ax_row.set_ylim(0, 1.15)
        # Annotate raw values on top of bars
        for i, v in enumerate(cls_vals):
            ax_row.text(i - width/2, cls_norm[i] + 0.02, f'{v:.3g}',
                         ha='center', fontsize=7)
        for i, v in enumerate(qnt_vals):
            ax_row.text(i + width/2, qnt_norm[i] + 0.02, f'{v:.3g}',
                         ha='center', fontsize=7)

    plt.tight_layout()
    plt.savefig(RESULTS / 'comparison_classical_vs_quantum.png', dpi=140, bbox_inches='tight')
    plt.show()
    print('Saved figure to results/comparison_classical_vs_quantum.png')
elif not paired_keys:
    print('No matching (variant, assets) pairs across classical+quantum yet — '
          'run notebook 01 and 02 with the same TICKERS and MODEL_VARIANT to enable this.')

## 4. Distribution side-by-side (for the slide)

For one chosen (variant, assets) pair, load the saved samples and plot histograms with all available models on the same axes. This is the visual workhorse for the talk.

In [ ]:
# Pick what to show. Defaults to the multivariate WGAN-GP comparison if available.
PICK_VARIANT = 'wgan_gp'
PICK_ASSETS  = 'SSMI_GDAXI'   # or 'SSMI', etc. — must match a folder suffix

if df.empty:
    print('No data — skipping.')
else:
    sub = df[(df['variant'] == PICK_VARIANT) & (df['assets'] == PICK_ASSETS)]
    if sub.empty:
        print(f'No runs with variant={PICK_VARIANT}, assets={PICK_ASSETS}. '
              f"Available: {df[['variant','assets']].drop_duplicates().to_dict('records')}")
    else:
        # Find tickers from the metrics file
        first = sub.iloc[0]
        first_metrics = json.load(open(RESULTS / first['run'] / 'metrics.json'))
        tickers = first_metrics['tickers']
        n_assets = len(tickers)

        # Real returns: take from any run (they're identical — same train/test split)
        real = np.load(RESULTS / first['run'] / 'real_returns_test.npy')
        if real.ndim == 2: real = real[..., None]

        fig, axes = plt.subplots(n_assets, 1, figsize=(9, 3.2 * n_assets), squeeze=False)
        for i, ticker in enumerate(tickers):
            ax = axes[i, 0]
            real_flat = real[:, :, i].ravel()
            bins = np.linspace(real_flat.min(), real_flat.max(), 80)
            ax.hist(real_flat, bins=bins, alpha=0.4, density=True, label='real',
                    color='black', histtype='stepfilled')

            for _, row in sub.iterrows():
                fake = np.load(RESULTS / row['run'] / 'fake_returns.npy')
                if fake.ndim == 2: fake = fake[..., None]
                fake_flat = fake[:, :, i].ravel()
                tag = f"{row['family']}"
                if pd.notna(row['n_qubits']):
                    tag += f" (q={int(row['n_qubits'])})"
                ax.hist(fake_flat, bins=bins, alpha=0.5, density=True,
                        label=tag, histtype='step', linewidth=2)
            ax.set_title(f'{ticker} — return distribution ({PICK_VARIANT}, {PICK_ASSETS})')
            ax.set_xlabel('log-return'); ax.legend()
        plt.tight_layout()
        plt.savefig(RESULTS / f'distributions_{PICK_VARIANT}_{PICK_ASSETS}.png',
                    dpi=140, bbox_inches='tight')
        plt.show()
        print(f'Saved to results/distributions_{PICK_VARIANT}_{PICK_ASSETS}.png')

## 5. Cross-asset correlation comparison

If the chosen comparison is multivariate, show how each model recovers the cross-asset correlation.

In [ ]:
if not df.empty and not sub.empty and n_assets >= 2:
    runs_to_show = sub.copy()
    n_runs = len(runs_to_show)

    fig, axes = plt.subplots(1, n_runs + 1, figsize=(3.2 * (n_runs + 1), 3.2))
    real_corr = np.array(first_metrics['dependence']['real_corr']
                         if 'dependence' in first_metrics
                         else first_metrics['correlation']['real_corr'])
    im = axes[0].imshow(real_corr, vmin=-1, vmax=1, cmap='RdBu_r')
    axes[0].set_title('real')
    axes[0].set_xticks(range(n_assets)); axes[0].set_yticks(range(n_assets))
    axes[0].set_xticklabels(tickers, rotation=45); axes[0].set_yticklabels(tickers)
    for i in range(n_assets):
        for j in range(n_assets):
            axes[0].text(j, i, f'{real_corr[i,j]:.2f}', ha='center', va='center',
                          fontsize=10)

    for k, (_, row) in enumerate(runs_to_show.iterrows()):
        m = json.load(open(RESULTS / row['run'] / 'metrics.json'))
        fake_corr = np.array(m['dependence']['fake_corr']
                              if 'dependence' in m
                              else m['correlation']['fake_corr'])
        ax = axes[k + 1]
        ax.imshow(fake_corr, vmin=-1, vmax=1, cmap='RdBu_r')
        tag = f"{row['family']}"
        if pd.notna(row['n_qubits']):
            tag += f" (q={int(row['n_qubits'])})"
        ax.set_title(tag)
        ax.set_xticks(range(n_assets)); ax.set_yticks(range(n_assets))
        ax.set_xticklabels(tickers, rotation=45); ax.set_yticklabels([])
        for i in range(n_assets):
            for j in range(n_assets):
                ax.text(j, i, f'{fake_corr[i,j]:.2f}', ha='center', va='center',
                         fontsize=10)
    fig.colorbar(im, ax=axes, fraction=0.025)
    plt.savefig(RESULTS / f'correlation_{PICK_VARIANT}_{PICK_ASSETS}.png',
                dpi=140, bbox_inches='tight')
    plt.show()
    print(f'Saved to results/correlation_{PICK_VARIANT}_{PICK_ASSETS}.png')
elif n_assets < 2:
    print(f'Only {n_assets} asset(s) — correlation plot skipped (need ≥2).')

## 6. Qubit-scaling plot

If the qubit-scan was run in notebook 02, plot how each metric scales with `n_qubits`. Looks for either a `quantum_scan_*` folder, or for multiple quantum runs with the same (variant, assets) but different `n_qubits`.

In [ ]:
if df.empty:
    print('No data.')
else:
    # Method 1: standalone scan files (preferred)
    scan_files = list(RESULTS.glob('quantum_scan_*/scan_results.json'))
    scan_data = []
    if scan_files:
        for sf in scan_files:
            with open(sf) as f:
                arr = json.load(f)
            for r in arr:
                scan_data.append({
                    'source': sf.parent.name,
                    'n_qubits': r.get('n_qubits'),
                    'ks': r.get('ks_statistic_overall'),
                    'training_time_sec': r.get('training_time_sec'),
                    'kurtosis_fake': np.mean([v['kurtosis']
                                              for v in r.get('fake_per_asset', {}).values()])
                                      if r.get('fake_per_asset') else np.nan,
                    'kurtosis_real': np.mean([v['kurtosis']
                                              for v in r.get('real_per_asset', {}).values()])
                                      if r.get('real_per_asset') else np.nan,
                })
    # Method 2: fall back to multiple quantum entries with same (variant, assets)
    if not scan_data:
        for (variant, assets), grp in df[df['family'] == 'quantum'].groupby(
                ['variant', 'assets']):
            grp = grp.sort_values('n_qubits')
            if grp['n_qubits'].nunique() >= 2:
                for _, r in grp.iterrows():
                    scan_data.append({
                        'source': f'{variant}_{assets}',
                        'n_qubits': r['n_qubits'],
                        'ks': r['ks'],
                        'training_time_sec': r['training_time_sec'],
                        'kurtosis_fake': r.get('kurtosis_fake'),
                        'kurtosis_real': r.get('kurtosis_real'),
                    })

    scan_df = pd.DataFrame(scan_data)
    if scan_df.empty:
        print('No scaling data found. Run notebook 02 with RUN_QUBIT_SCAN=True, '
              'or run notebook 02 multiple times with different N_QUBITS.')
    else:
        print(scan_df.to_string(index=False))

In [ ]:
if not df.empty and not scan_df.empty:
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    for src, grp in scan_df.groupby('source'):
        grp = grp.sort_values('n_qubits')
        ax[0].plot(grp['n_qubits'], grp['ks'], 'o-', label=src)
        ax[1].plot(grp['n_qubits'], grp['kurtosis_fake'], 'o-', label=src)
        ax[2].plot(grp['n_qubits'], grp['training_time_sec'], 'o-', label=src)
    # Real-kurtosis reference line
    if scan_df['kurtosis_real'].notna().any():
        ax[1].axhline(scan_df['kurtosis_real'].dropna().iloc[0],
                       color='C1', ls='--', label='real (target)')

    ax[0].set_xlabel('n_qubits'); ax[0].set_ylabel('KS distance')
    ax[0].set_title('Distribution match (lower = better)'); ax[0].legend(fontsize=8)
    ax[1].set_xlabel('n_qubits'); ax[1].set_ylabel('avg kurtosis')
    ax[1].set_title('Tail behavior'); ax[1].legend(fontsize=8)
    ax[2].set_xlabel('n_qubits'); ax[2].set_ylabel('training time (s)')
    ax[2].set_title('Resource cost'); ax[2].set_yscale('log'); ax[2].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(RESULTS / 'qubit_scaling.png', dpi=140, bbox_inches='tight')
    plt.show()
    print('Saved to results/qubit_scaling.png')

## 7. One-line summary statements for the slides

Auto-generates the kind of bullet points you'd put on a results slide. Reads the table and constructs the sentences.

In [ ]:
if df.empty:
    print('No data — skipping.')
elif paired_keys:
    for (variant, assets) in paired_keys:
        sub = df[(df['variant'] == variant) & (df['assets'] == assets)]
        cls = sub[sub['family'] == 'classical'].iloc[0]
        qnt = sub[sub['family'] == 'quantum'].sort_values('n_qubits',
                                                            na_position='first').iloc[-1]
        print(f"\n--- assets={assets} | variant={variant} ---")
        print(f"  Generator params:    classical={cls['n_params_G']}, "
              f"quantum={qnt['n_params_G']}")
        if pd.notna(qnt['n_qubits']):
            q_quant = int(qnt['n_qubits']) * int(qnt['n_layers']) if pd.notna(qnt['n_layers']) else None
            print(f"    (quantum: {int(qnt['n_qubits'])} qubits x {int(qnt['n_layers']) if pd.notna(qnt['n_layers']) else '?'} layers "
                  f"= {q_quant if q_quant is not None else '?'} quantum params)")
        print(f"  Training time:       classical={cls['training_time_sec']:.1f}s, "
              f"quantum={qnt['training_time_sec']:.1f}s  "
              f"(quantum slower by {qnt['training_time_sec']/max(cls['training_time_sec'], 0.1):.1f}x)")
        print(f"  KS distance:         classical={cls['ks']:.4f}, quantum={qnt['ks']:.4f}")
        if 'bures_wasserstein' in cls and pd.notna(cls['bures_wasserstein']):
            print(f"  Bures-Wasserstein:   classical={cls['bures_wasserstein']:.6f}, "
                  f"quantum={qnt['bures_wasserstein']:.6f}")
        if 'frobenius_corr' in cls and pd.notna(cls['frobenius_corr']):
            print(f"  Frobenius (corr):    classical={cls['frobenius_corr']:.4f}, "
                  f"quantum={qnt['frobenius_corr']:.4f}")
        print(f"  Std rel. error:      classical={cls['std_rel_err']:.3f}, "
              f"quantum={qnt['std_rel_err']:.3f}")
        print(f"  Kurtosis gap:        classical={cls['kurtosis_gap']:.2f}, "
              f"quantum={qnt['kurtosis_gap']:.2f}  (real={cls['kurtosis_real']:.2f})")

## What's been saved for the presentation

All outputs live in `results/`:

- `summary.csv` — full table of every run, copy-paste into slides
- `comparison_classical_vs_quantum.png` — bar chart, one panel per (variant, assets) pair
- `distributions_<variant>_<assets>.png` — return histograms with all models overlaid
- `correlation_<variant>_<assets>.png` — heatmap of cross-asset correlations
- `qubit_scaling.png` — scaling experiment results (if available)

Suggested 10-minute talk structure:

1. **(2 min)** problem framing + univariate vs multivariate, theoretical motivation for quantum
2. **(1 min)** GAN architecture (with the rendered circuit from notebook 02)
3. **(2 min)** evaluation methodology + metrics (KS / kurtosis / Bures-Wasserstein / Frobenius)
4. **(2 min)** results — distribution plot + correlation plot
5. **(1 min)** qubit-scaling plot
6. **(1 min)** honest framing: what worked, what didn't, why
7. **(1 min)** future work + Q&A